# 

In [1]:
import cdsaxs
dict[cdsaxs]

ModuleNotFoundError: No module named 'CDSAXS_base_model'

In [2]:
%load_ext autoreload
%autoreload 2
import os
import numpy as np

# Import fitting modules from the installed cdsaxs package
from cdsaxs.Fitting.CDSAXS_base_model import CDSAXS_Model
from cdsaxs.Fitting.Trapezoid_model import TrapezoidModel

import matplotlib.pyplot as plt
from scipy.optimize import differential_evolution
import pandas as pd
import time

import warnings
warnings.filterwarnings('ignore', category=RuntimeWarning) # there is a runtime warning that comes from nans/zeros is the freeform trapezoid calculation turn this on to ignore that warning

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


ModuleNotFoundError: No module named 'CDSAXS_base_model'

# Initialize Single Layer Model

In [ ]:

geometry='trapezoid' # These don't do anything yet, but the goal is to generalize the code so that you can give it a geomeotry (i.e. trapezoidal, cylindrical)
model='symmetric' # similarly the goal is to give it a model that you can then simulate


# Setup Initial Model positions
model_params = {
    'layers': 1,  # Number of layers
    'trapezoids': [
        {'width': 810, 'height': 550},
        {'width': 83, 'height': 0},
    ],
    'DW': 15,  # Debye-Waller factor
    'I0': 0.000001,  # Intensity scaling
    'Bk': 0.5  # Background
}

# setup parameter range for optimizations
model_params['optimization'] = {
    'trap_0_width': {'min': 700, 'max': 1000},
    'trap_0_height': {'min': 500, 'max': 600},
    'trap_1_width': {'min': 30, 'max': 120},
    'DW': {'min': 1, 'max': 50, 'default': 15}
    # Only parameters you want to optimize need to be included
}
# setup model based on parameters and optimization parameters
Model1 = TM.TrapezoidModel(
    model='single_material',
    layers=model_params['layers'],
    model_params=model_params
)

# add data to the model
Model1.importCDSAXS_GUI('75s_30nm_200nmpitch_IvsQz.csv')

# plot the initial structure
Model1.plot_structure()
# plot the initial Simulated data compared to the experimental data
Model1.PlotQzCut()







# Optimize Single Layer Model

In [ ]:
Model1.CDSAXS_DiffEvolution()

# Generate a new model from the previous best fit

In [ ]:
Model2=Model1.add_layer_at_percentage(50)
Model2.plot_structure()



In [ ]:
Model2.simulate_structure()
Model2.PlotQzCut()

# Parameter Sweep single layer model

In [ ]:
results_1d=Model1.parameter_sweep_1d(sweep_param='trap_0_height', sweep_range=(550,570), n_points=5, plot_results=True, verbose=True, optimization_kwargs={'workers':8})

In [ ]:
Model1.apply_optimal_parameters_1d(results_1d)
Model1.model_params

# 2 Parameter sweep

In [ ]:
sweep_parameters=('trap_0_width','DW')
sweep_ranges=((800,900),(20,30))
sweep_points=(5,5)
result_2d=Model1.parameter_sweep_2d(sweep_parameters,sweep_ranges,sweep_points)

In [ ]:
Model1.apply_optimal_parameters_2d(result_2d)

Model1.plot_structure()
Model1.PlotQzCut()

# Batch Initialization example
## This does not do a parameter sweep, it initializes multiple fits

In [ ]:
init_params = {
    'trap_0_width': {'min': 800, 'max': 950, 'n_points': 3},
    'DW': {'min': 10, 'max': 30, 'n_points': 2}
}

results = Model1.batch_initialize_and_fit(init_params)
Model1.display_top_fits(results, n_top=10)


In [ ]:
Model1.plot_fit_comparison(results, fit_ranks=[1, 2, 5])
Model1.apply_batch_fit_result(results, rank=1)  # Apply best fit

# 2 Layer Model

In [ ]:


geometry='trapezoid' # These don't do anything yet, but the goal is to generalize the code so that you can give it a geomeotry (i.e. trapezoidal, cylindrical)
model='symmetric' # similarly the goal is to give it a model that you can then simulate
model_params = {
    'layers': 2,  # Number of layers
    'trapezoids': [
        {'width': 810, 'height': 530},
        {'width': 100, 'height': 20},
        {'width': 83, 'height': 0},
    ],
    'DW': 15,  # Debye-Waller factor
    'I0': 0.000001,  # Intensity scaling
    'Bk': 0.5  # Background
}
model_params['optimization'] = {
    'trap_0_width': {'min': 700, 'max': 1000, 'default': 810},
    'trap_0_height': {'min': 500, 'max': 600, 'default': 550},
    'trap_1_width': {'min': 30, 'max': 120, 'default': 100},
    'trap_1_height': {'min': 10, 'max': 50, 'default': 20},
    'trap_2_width': {'min': 30, 'max': 120, 'default': 82},
    'DW': {'min': 1, 'max': 50, 'default': 15}
    # Only parameters you want to optimize need to be included
}

Model2 = TM.TrapezoidModel(
    model='single_material',
    layers=model_params['layers'],
    model_params=model_params
)


Model2.importCDSAXS_GUI('75s_30nm_200nmpitch_IvsQz.csv')
Model2.plot_structure()
# # Plot specific Qz cut
Model2.PlotQzCut()







In [ ]:
Model2.CDSAXS_DiffEvolution(**{'workers':4})

In [ ]:
Model2.model_params

# 3 layer model

In [ ]:
Model3=Model2.add_layer_at_percentage(50)
Model3.plot_structure()

In [ ]:
Model3.CDSAXS_DiffEvolution()

In [ ]:
result_2d=Model3.parameter_sweep_2d(('trap_0_width','DW'),((800,900),(20,30)),(10,10))